In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Horizontal Visibility Graph Network — Italy INGV Catalog (1985-2025)

Run as a script  : python ITALY_network_HVG.py
Convert to notebook: python convert_to_notebook.py ITALY_network_HVG.py notebooks/ITALY_network_HVG.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.network import build_hvg_network
from src.metrics import estimate_gamma_mle, test_power_law, measure_pa_growing_graph, plot_preferential_attachment
from src.assortativity import compute_assortativity, plot_assortativity, plot_knn, plot_rich_club
from src.centrality import (
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
    compute_bb_fitness_events,
    plot_bb_fitness,
    plot_bb_fitness_theory,
    plot_bb_fitness_geo,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_directed_louvain,
    run_hdbscan_spectral,
    run_hdbscan_geo,
    run_bigclam,
    compare_partitions,
    plot_partition_scores,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    plot_community_geo,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)

CUT_YEAR = 1985

# HVG model parameter
HVG_MAX_LOOK_AHEAD = 10_000   # safety cap; early termination fires faster than NVG

SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "hvg")

## Data Loading

The Italy INGV catalog covers earthquakes $M \geq 1.5$ from 1985 to 2025 in the
bounding box $[34°\text{N}, 48°\text{N}] \times [3°\text{E}, 22°\text{E}]$,
downloaded from the INGV FDSN web service.

The HVG is constructed **purely from the chronological magnitude sequence** —
spatial coordinates (latitude, longitude, depth) are stored as node attributes
for geographic visualisation but play no role in edge formation.  Unlike the
BP/ZBZ models, no spatial projections or distance computations are required.

Events are sorted by origin time and indexed $0, 1, \ldots, N-1$ to define the
time-series ordering $\{m_0, m_1, \ldots, m_{N-1}\}$ on which the HVG is built.

**Expected output:** ~215,000 rows spanning 1985–2025.  The catalog is the same
as used in the ABE, BP, and TL analyses — only the network construction step differs.

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

## Network Construction

The **Horizontal Visibility Graph** (HVG) is a stricter variant of the
Natural Visibility Graph (NVG / Telesca-Lovallo 2012).  Two events $i$ and $j$
($i < j$ in time) are connected iff every intermediate event $k$ satisfies:

$$\text{HVG:}\quad (i, j) \in E \iff \forall k \in (i,j):\; m_k < \min(m_i, m_j)$$

Geometrically, the horizontal bar at height $\min(m_i, m_j)$ must clear all
intermediate magnitudes — a strictly more demanding condition than the NVG slope
test, which only requires the line connecting $(i, m_i)$ and $(j, m_j)$ to lie
above all intermediate points $k$.

The HVG construction admits a true $O(n)$ algorithm with early termination:
once the running maximum of intermediates $\max_{i < k < j} m_k$ reaches $m_i$,
no further $j$ can be visible from $i$ and the inner loop exits immediately.
Every adjacent pair $(i, i+1)$ is trivially visible (no intermediates exist),
so the HVG is **always connected** — it always contains the path $0-1-\cdots-(N-1)$.

**Sparsity:** HVG average degree $\langle k \rangle \approx 2$–$4$ vs NVG
$\approx 4$–$8$.  The difference arises because the NVG allows slope-visible
long-range edges blocked by the HVG horizontal criterion.

**Seismological interpretation:** HVG hubs are the magnitude-dominant mainshocks
that suppress horizontal visibility for extended windows before and after them.
A large mainshock at index $i$ blocks all events within the range where its
magnitude exceeds every intermediate — seismologically, it dominates the
amplitude landscape of entire aftershock sequences.

*References:*
Luque, B., Lacasa, L., Ballesteros, F., & Luque, J. (2009).
Horizontal visibility graphs: Exact results for random time series.
*Physical Review E*, 80(4), 046103.
Telesca, L., & Lovallo, M. (2012). *EPL*, 97, 50002.

**Expected output:** ~215,000 nodes; ~270,000–400,000 edges; $\langle k \rangle \approx 2.5$–$3.5$;
build time < 30 seconds on a modern CPU.  Min degree = 1 (endpoints), max degree
could reach thousands for the largest mainshocks (Amatrice M6.2 2016, L'Aquila M6.3 2009).

In [ ]:
print(f"Building HVG (max_look_ahead={HVG_MAX_LOOK_AHEAD})…")
t_build = time.time()
G = build_hvg_network(df_net, max_look_ahead=HVG_MAX_LOOK_AHEAD)
print(f"Build time: {time.time() - t_build:.1f}s")
print(f"Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")
print(f"Average degree: {2*G.number_of_edges()/G.number_of_nodes():.2f}")
print(f"Min degree: {min(d for _, d in G.degree())}  "
      f"Max degree: {max(d for _, d in G.degree()):,}")

## Hub Map — 2D

High-degree events in the HVG are magnitude-dominant mainshocks that maintain
unobstructed horizontal lines of sight to many other events in the time series.
The HVG condition is stricter than NVG: only events that exceed every intermediate
magnitude in an extended window can accumulate many direct horizontal links.

These correspond to the global and regional magnitude maxima in the catalog:
the L'Aquila 2009 ($M_W$ 6.3), Amatrice 2016 ($M_W$ 6.2), and Irpinia 1980
($M_W$ 6.9) mainshocks are expected to dominate.

Compared to TL (NVG) hubs, HVG hubs are **fewer and more extreme** — the
stricter horizontal condition means that only the tallest amplitude peaks in the
40-year sequence accumulate many direct links.

**Expected output:** Map showing 100 hub events concentrated near the major fault
systems of central Italy (Apennines) and Southern Italy.  Top hub degree likely
$> 1{,}000$ for the largest mainshocks, falling sharply below 100 for the remaining
95+ hubs — a pronounced degree inequality reflecting the heavy-tailed magnitude
distribution.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "event_idx": n,
        "degree":    G.degree(n),
        "lat":       G.nodes[n]["lat"],
        "lon":       G.nodes[n]["lon"],
        "magnitude": G.nodes[n]["magnitude"],
    }
    for n in G.nodes()
])
df_hubs = df_nodes.nlargest(100, "degree").copy()
print(f"Top 100 hubs: degree range [{df_hubs['degree'].min():.0f}, {df_hubs['degree'].max():.0f}]")

# Sort ascending so high-degree markers render on top in plotly
df_hubs = df_hubs.sort_values("degree")

# Scale size to [8, 35] range
_deg_min = df_hubs["degree"].min()
_deg_max = df_hubs["degree"].max()
df_hubs["marker_size"] = 8 + 27 * (df_hubs["degree"] - _deg_min) / max(_deg_max - _deg_min, 1)

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="degree", size="marker_size",
    size_max=35, opacity=0.85,
    color_continuous_scale="inferno",
    map_style="carto-positron",
    hover_name="event_idx",
    hover_data={"magnitude": ":.2f", "degree": True, "marker_size": False},
    title="HVG Network Hubs: Top 100 Highest-Degree Events — Italy",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "hvg_hub_map_2d_italy")
fig.show()

## Degree Distribution — Linear

Linear-scale histogram of node degrees, truncated at $k = 50$.
The HVG minimum degree is 1 for the first and last events (only one neighbour),
and 2 for all other events (always connected to both immediate neighbours).

The theoretical HVG degree distribution for an i.i.d. uniform series is
$P(k) = (1/3)(2/3)^{k-2}$, a **geometric distribution** with mean $\langle k \rangle = 4$
(Luque et al. 2009).  On a log-linear scale this is a straight line (exponential
decay), not a power law.

Seismic catalogs deviate from this baseline because magnitude is not i.i.d.:
aftershock clustering creates extended runs of small-magnitude events that are
invisible from far-away large events, suppressing short-lag connectivity and
concentrating links around mainshocks.

**Expected output:** A steeply decaying histogram with the bulk of events at
$k = 2$–$5$ and a thin tail extending to $k > 100$ for the largest mainshocks.
Roughly 30–50% of all events should have degree exactly 2 (adjacent-pair links only),
reflecting the background of low-magnitude events invisible beyond one step.

In [ ]:
plot_degree_distribution_linear(G, "HVG Italy", max_degree=50, weighted=False)

## Degree Distribution — Log-Log

Log-log scatter of $P(k)$ vs $k$.  For an i.i.d. series the HVG distribution
is geometric (exponential on a log-linear plot), not a power law — a straight line
on a log-linear plot, concave on a log-log plot.

A power-law tail on a log-log plot signals non-i.i.d. structure.  In seismic
catalogs this arises from the heavy-tailed magnitude distribution (Gutenberg-Richter):
rare large events produce very high degrees by dominating the amplitude landscape
over extended time windows, far beyond what an i.i.d. geometric model predicts.

Compared to TL (NVG), the HVG tail is expected to be **lighter**: the stricter
horizontal condition removes many long-range links that the NVG retains via slope
visibility, so the degree distribution is less heavy-tailed.

**Expected output:** Faster-than-power-law rolloff for most of the range, with
a possibly linear segment at high $k$ driven by the largest mainshocks.
The log-log plot will appear more curved than the corresponding NVG plot.

In [ ]:
analyze_degree_distribution(G, "HVG Italy")

## Degree Distribution — Log Binning

Logarithmic binning (Clauset, Shalizi & Newman 2009) places exponentially wider
bins across the degree axis, stabilising the sparse high-$k$ tail.  The slope of
the resulting log-log line gives a visual estimate of the apparent tail exponent
before MLE fitting.

Because the HVG tail is lighter than the NVG, the fitted slope from log-binned
data is expected to be **steeper** (larger $\gamma$) than the TL estimate —
indicating a faster-decaying tail.  Any apparent power-law segment is likely
confined to a narrow range of $k$ in the upper tail.

**Expected output:** Log-binned scatter with a fitted line.  Slope likely
$\hat{\gamma}_{\text{log-bin}} \approx 3.5$–$6.0$, steeper than NVG.

In [ ]:
analyze_degree_distribution_log_binning(G, "HVG Italy", k_min_fit=4)

## Degree Distribution — CCDF

The complementary CDF $P(K \geq k) = 1 - F(k)$ avoids binning artefacts entirely.
For a power-law distribution the CCDF is linear on a log-log scale with slope
$-(\gamma - 1)$.  A **concave** CCDF is consistent with a geometric or
stretched-exponential distribution — the expected i.i.d. baseline for the HVG.

Any straight segment at the upper tail indicates that the heavy-tailed magnitude
distribution is creating approximately power-law connectivity among the top
mainshocks, even though the bulk of the distribution remains geometric.

**Expected output:** Predominantly concave CCDF (faster than power-law decay),
with a possible near-linear segment for $k > 100$ corresponding to the handful
of major mainshocks in the catalog.  The CCDF will drop more steeply than the
corresponding TL (NVG) CCDF at the same $k$.

In [ ]:
plot_ccdf_with_fit(G, "HVG Italy", k_min_fit=4)

## MLE Gamma

Maximum-likelihood estimate of the tail exponent (Clauset, Shalizi & Newman 2009):

$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$

where $k_{\min}$ is the lower cutoff estimated by minimising the Kolmogorov-Smirnov
distance between the empirical tail and the fitted power law.

The CSN likelihood-ratio test compares the power-law against an exponential
alternative.  For a purely geometric (i.i.d. uniform) series, the exponential
model always wins ($R < 0$).  A positive $R$ with $p < 0.05$ would indicate that
Gutenberg-Richter statistics introduce a heavier-than-geometric HVG tail —
direct evidence that seismic magnitude clustering breaks the i.i.d. assumption.

*Reference:* Luque et al. (2009) proved analytically that the i.i.d. HVG
exponent is $\gamma_{\text{theoretical}} = +\infty$ (geometric, no power law),
so any empirical power-law detection on a seismic HVG is attributable entirely
to non-i.i.d. magnitude structure.

**Expected output:** $\hat{\gamma} \approx 3.5$–$6.0$ (steeper than NVG
$\approx 2.5$–$4.0$); the CSN test may favour the exponential model ($R < 0$)
for most of the distribution, with the verdict flipping to power-law only if
the upper tail sample is large enough to detect mainshock clustering.

In [ ]:
degs      = [d for _, d in G.degree()]
gamma_hvg = estimate_gamma_mle(degs, k_min=4)
print(f"MLE γ (degree, k_min=4): {gamma_hvg:.3f}")

result_hvg = test_power_law(degs, k_min=4)
print(f"  CSN test: γ={result_hvg['gamma']} (σ={result_hvg['sigma']})  "
      f"R={result_hvg['R']}  p={result_hvg['p_value']}  → {result_hvg['verdict']}")

## Preferential Attachment

The HVG is the strictest visibility graph: event $j$ sees $i$ only when no
intermediate event exceeds $\min(m_i, m_j)$.  The resulting network has
a theoretically known degree distribution for i.i.d. series
($P(k) = (1/3)(2/3)^{k-2}$, geometric), which provides a natural null.

The preferential attachment kernel $\pi(k) \propto k^{\alpha}$ is measured
with the same *batch arrival* convention used for the NVG:

$$\pi(k) = \frac{\sum_{i:\,k_i(t)=k} \Delta k_i}{\#\{i:\,k_i(t)=k\}}$$

Because the HVG is sparser than the NVG (no long-range connections), the
degree distribution truncates at lower $k$, giving fewer data points in the
high-$k$ tail.  Sub-linear attachment ($\alpha < 1$) is expected when the
connectivity of large events is constrained by the strict horizontal visibility
criterion — once a large mainshock has accumulated several neighbours, further
events must exceed it to "see through" to those neighbours, concentrating
new attachments on the mainshock itself.

*Reference:* Jeong H., Néda Z. & Barabási A.-L. (2003). Measuring preferential
attachment in evolving networks. *Europhysics Letters*, 61(4), 567–572.

In [ ]:
print("Measuring preferential attachment kernel (HVG growing graph)…")
ks_pa, pi_k_pa, alpha_pa = measure_pa_growing_graph(G, df_net)
plot_preferential_attachment(ks_pa, pi_k_pa, alpha_pa, title="HVG Italy (HVG PA)")

## Macroscopic Metrics

The HVG is **always connected** (adjacent pairs are trivially linked), so no
giant-component extraction is needed — the full graph is the connected component.

Average degree $\langle k \rangle \approx 2$–$4$ is lower than NVG ($\approx 4$–$8$)
because the horizontal condition eliminates slope-visible but horizontally-obstructed
long-range links.  This sparsity has downstream effects on clustering and path length.

**Clustering coefficient** $C$ measures the fraction of triples $(i, j, k)$ where
all three events can mutually see each other horizontally.  In an i.i.d. HVG,
$C = 0$ (no triangles allowed by the horizontal condition — if $i$ sees $j$
and $j$ sees $k$, the intermediate $j$ blocks $i$ from seeing $k$ directly unless
$m_j < \min(m_i, m_k)$, which contradicts the edge criterion).  Any empirical
$C > 0$ reflects the non-i.i.d. structure of seismic magnitudes.

**Average path length** $L$ in a connected path-like graph grows as $O(N)$ in the
worst case.  The HVG long-range edges (mainshock hubs) provide shortcuts, reducing
$L$ below the pure path-graph expectation but not as dramatically as the NVG.

**Small-world** signature $C \gg C_{\text{ER}}$ with $L \approx L_{\text{ER}}$
is expected to be **weaker** for HVG than NVG: sparser structure reduces local
clustering while slightly increasing path lengths.

*ER baselines:* $C_{\text{ER}} = \langle k \rangle / N$; $L_{\text{ER}} = \ln N / \ln \langle k \rangle$.

**Expected output:** $C \approx 0.05$–$0.20$; $C / C_{\text{ER}} \approx 10$–$100\times$;
$L \approx 30$–$80$ hops (longer than NVG ≈ 15–35); $L / L_{\text{ER}} \approx 3$–$10\times$.

In [ ]:
N_nodes = G.number_of_nodes()
M_edges = G.number_of_edges()
avg_k   = 2 * M_edges / N_nodes
print(f"Nodes: {N_nodes:,}  Edges: {M_edges:,}  ⟨k⟩ = {avg_k:.2f}")

print("Computing clustering coefficient…")
avg_c = nx.average_clustering(G)
c_er  = avg_k / N_nodes
print(f"Avg clustering C = {avg_c:.4f}  (ER expected: {c_er:.4f}  ratio: {avg_c/max(c_er,1e-12):.1f}×)")

print("Approximating average path length (500 random seeds)…")
rng       = np.random.default_rng(42)
seeds_apl = list(rng.choice(N_nodes, size=min(500, N_nodes), replace=False))
apl_vals  = []
for s in seeds_apl:
    lengths = nx.single_source_shortest_path_length(G, s)
    apl_vals.extend(lengths.values())
avg_L = np.mean(apl_vals)
l_er  = np.log(N_nodes) / np.log(avg_k) if avg_k > 1 else float("nan")
print(f"Approx avg path length L ≈ {avg_L:.2f}  (ER expected: {l_er:.2f})")
print(f"Small-world: C/C_ER = {avg_c/max(c_er,1e-12):.1f}×,  L/L_ER = {avg_L/max(l_er,1e-12):.2f}×")

## Centrality Analysis

Nine centrality measures on the HVG undirected graph identify magnitude-structurally
important events under the stricter horizontal visibility criterion.

- **Degree centrality:** captures direct horizontal visibility — events that
dominate amplitude over the widest time window.
- **PageRank:** propagates influence through the HVG chain, up-weighting events
connected to other influential hubs.  On the HVG, this amplifies the
effect of mainshocks at the centre of dense aftershock clusters.
- **Betweenness:** identifies events lying on the unique shortest path between
many pairs — amplitude "bottlenecks" that separate distinct seismic episodes.
Because the HVG is sparser (closer to a path graph) than the NVG, betweenness
is more concentrated: bridge events carry disproportionately high load.
- **Eigenvector / Katz:** weight connections by the importance of neighbours,
rewarding events connected to other high-magnitude mainshocks.
- **HITS (hubs/authorities):** in a temporal visibility chain, hubs point to
many events while authorities are pointed to by many.  On an undirected HVG
the distinction is symmetric, but HITS still reveals structural differences
from degree in the presence of high-degree subgraphs.
- **Harmonic centrality**: $C_H(v) = \sum_{u \neq v} 1/d(u,v)$, summing inverse
shortest-path distances. Because the HVG is always connected (each node links
to at least its two temporal neighbours), harmonic centrality here provides a
smooth alternative to closeness — events with high harmonic reach sit near
the structural centre of the full 40-year magnitude series
(*Bavelas 1950; Rochat 2009*).
- **Clustering coefficient**: fraction of a node's neighbour pairs that are
mutually connected ($C_i = 2t_i / k_i(k_i-1)$). The HVG is denser than a
pure path but sparser than the NVG; non-trivial clustering indicates magnitude
episodes where the horizontal-visibility criterion is satisfied among several
consecutive events simultaneously — compact, homogeneous aftershock bursts
(*Watts & Strogatz 1998*).
- **Triangle count**: raw number of closed triangles per node. Confirms that the
HVG has non-chain topology; high counts flag the densest seismic episodes.

Centralities are computed inline because `compute_all_centralities` parses node
IDs as `"cx_cy_cz"` strings (ABE format), which does not apply to HVG integer
indices.

**Expected output:** Top-5 degree and betweenness lists dominated by the same
3–5 large mainshocks.  PageRank rankings may differ slightly, elevating events
that connect to other high-degree mainshocks rather than purely the highest-degree
single event.  Betweenness should be highly concentrated — the largest mainshock
may account for a disproportionate fraction of all shortest paths.

In [ ]:
print("Computing centralities for HVG network…")
_t0_cent = time.time()

G_nsl = G.copy()
G_nsl.remove_edges_from(nx.selfloop_edges(G_nsl))
_N    = G.number_of_nodes()

deg_cent = nx.degree_centrality(G)
print(f"  Degree done ({time.time()-_t0_cent:.1f}s)")

pr_cent = nx.pagerank(G, weight="weight")
print(f"  PageRank done ({time.time()-_t0_cent:.1f}s)")

bet_cent = nx.betweenness_centrality(G, k=min(500, _N), seed=42, normalized=True)
print(f"  Betweenness done ({time.time()-_t0_cent:.1f}s)")

try:
    eig_cent = nx.eigenvector_centrality(G, weight="weight", max_iter=500, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    eig_cent = nx.eigenvector_centrality_numpy(G, weight="weight")
print(f"  Eigenvector done ({time.time()-_t0_cent:.1f}s)")

_max_deg    = max(d for _, d in G.degree())
_alpha_katz = 0.85 / _max_deg
try:
    katz_cent = nx.katz_centrality(
        G, alpha=_alpha_katz, weight="weight", normalized=True, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    katz_cent = nx.katz_centrality_numpy(G, alpha=_alpha_katz, weight="weight")
print(f"  Katz done ({time.time()-_t0_cent:.1f}s)")

try:
    hits_hub, hits_auth = nx.hits(G_nsl, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    _zeros    = {n: 0.0 for n in G.nodes()}
    hits_hub  = _zeros.copy()
    hits_auth = _zeros.copy()
print(f"  HITS done ({time.time()-_t0_cent:.1f}s total)")

harm_cent = nx.harmonic_centrality(G)
print(f"  Harmonic done ({time.time()-_t0_cent:.1f}s)")

clust_cent = nx.clustering(G, weight="weight")
print(f"  Clustering done ({time.time()-_t0_cent:.1f}s)")
tri_count = nx.triangles(G)
print(f"  Triangles done ({time.time()-_t0_cent:.1f}s)")

df_centrality = pd.DataFrame([
    {
        "cell_id":     n,
        "lat":         G.nodes[n]["lat"],
        "lon":         G.nodes[n]["lon"],
        "depth_km":    G.nodes[n]["depth_km"],
        "Degree":      deg_cent.get(n, 0.0),
        "PageRank":    pr_cent.get(n, 0.0),
        "Betweenness": bet_cent.get(n, 0.0),
        "Eigenvector": eig_cent.get(n, 0.0),
        "Katz":        katz_cent.get(n, 0.0),
        "HITS_Hub":    hits_hub.get(n, 0.0),
        "HITS_Auth":   hits_auth.get(n, 0.0),
        "Harmonic":    harm_cent.get(n, 0.0),
        "Clustering":  clust_cent.get(n, 0.0),
        "Triangles":   float(tri_count.get(n, 0)),
    }
    for n in G.nodes()
    if "lat" in G.nodes[n] and "lon" in G.nodes[n]
])

for metric, label in [
    ("Degree",      "Amplitude-Dominant Events"),
    ("PageRank",    "Propagation Influencers"),
    ("Betweenness", "Sequence Bottlenecks"),
    ("HITS_Hub",    "Visibility Hubs"),
    ("HITS_Auth",   "Visibility Authorities"),
    ("Harmonic",    "Topological Reach (Harmonic)"),
    ("Clustering",  "Triangle Density (Clustering)"),
    ("Triangles",   "Feedback Loops (Triangles)"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "HVG Italy")
plot_top_n_cells(df_centrality, top_n=10, title="HVG Italy")

## Centrality Geo Maps

Geographic projection of the top-10 events per centrality metric onto the Italy
basemap.

HVG hubs are amplitude-dominant mainshocks under the stricter horizontal
criterion, so the spatial distribution should be **more concentrated** than TL
(NVG) hubs — only events forming magnitude barriers in the time series appear,
and these coincide with the major historical mainshocks.  Expected dominance by:
Amatrice 2016 ($M_W$ 6.2, Apennines), L'Aquila 2009 ($M_W$ 6.3, central Italy),
and Irpinia 1980 ($M_W$ 6.9, southern Italy).

The **overlap map** scores each node by how many centrality metrics rank it in
the top 10.  High overlap (score ≥ 3) indicates a multi-metric hub — an event
that is simultaneously an amplitude barrier, a network bridge, and a PageRank
influencer.  These events are structurally indispensable: their removal would
fragment the long-range visibility structure of the magnitude sequence.

**Expected output:** Strong spatial convergence near the major Apennine ruptures.
The overlap map likely shows 1–3 events with multi-metric dominance —
far fewer than the TL (NVG) equivalent, reflecting the HVG's stricter selection.

In [ ]:
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

## Bianconi-Barabasi Fitness

The HVG has the sparsest connectivity of the two visibility paradigms: each event
connects only to events it can see horizontally (no taller intermediate events).
The fitness estimator $\hat{\beta}_i = \ln k_i(T) / \ln(T/t_i)$ applied to
this sparser graph gives a different view of seismogenic fitness than the NVG:

$$\hat{\beta}_i = \frac{\ln k_i(T)}{\ln(T/t_i)}$$

Because HVG degree is bounded (typically 2–6 per node for seismic series),
the dynamic range of $\hat{\beta}$ is narrow.  However, mainshocks with
exceptionally wide horizontal visibility (large gaps of lower-magnitude
activity before and after them) will achieve higher degree and hence higher
$\hat{\beta}$, identifying them as intrinsically prominent features of the
magnitude time series — independent of catalog duration.

The comparison with the NVG fitness distribution ($\gamma_{NVG}$ vs
$\gamma_{HVG}$) tests whether the stricter visibility criterion preserves
or distorts the fitness ranking of major mainshocks.

*References:* Bianconi G. & Barabási A.-L. (2001). *EPL* 54, 436–442;
*PRL* 86, 5632–5635.

In [ ]:
print("Computing Bianconi-Barabasi fitness (HVG events)…")
df_bb = compute_bb_fitness_events(G, df_net)
print(f"  {len(df_bb)} events with valid β̂; β range [{df_bb['fitness_beta'].min():.3f}, {df_bb['fitness_beta'].max():.3f}]")
plot_bb_fitness(df_bb, title="HVG Italy")
plot_bb_fitness_theory(df_bb, gamma=gamma_hvg, title="HVG Italy")
plot_bb_fitness_geo(
    df_bb,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0,
    bounds=_IT_BOUNDS, height=MAP_HEIGHT, width=MAP_WIDTH,
)

## Community Detection — Full Suite

Seven community-detection algorithms are applied to the undirected HVG.  Modularity

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

(Newman & Girvan 2004) is the primary optimisation target for graph-based methods,
but density- and affiliation-based methods operate on complementary representations.

**Seismological interpretation:** HVG community boundaries correspond to
magnitude-dominant events that block horizontal visibility between the events on
their left and right sides.  Each community is a contiguous segment of the time
series bounded by large mainshocks — "seismic episodes" separated by the largest
events in the catalog.  Because the HVG is **sparser** than the NVG (TL model),
fewer cross-episode links exist and community boundaries are sharper.  Louvain
modularity is expected to be higher for HVG than TL, and NMI across methods
should also be higher — the cleaner partition structure makes algorithm choices
less influential.

**Graph-based methods**
- **Louvain** (Blondel et al. 2008): greedy modularity optimisation; fast and
scalable but subject to resolution limit and stochastic variation.
- **Consensus Louvain** (Lancichinetti & Fortunato 2012): 100-run ensemble average
over co-assignment frequencies; stabilises stochastic partitions.
- **Spectral** (Von Luxburg 2007): Fiedler-vector bipartition applied recursively
to $k$ clusters; run on the full graph since the HVG is always connected.
- **InfoMap** (Rosvall & Bergstrom 2008): minimum-description-length flow
compression; may differ substantially from modularity methods on path-like graphs.

**Density-based methods**
- **HDBSCAN-Spectral** (Campello et al. 2013): applies HDBSCAN to the
$k$-dimensional spectral embedding of the normalised graph Laplacian.
The number of communities is determined entirely by the data — no $k$
pre-specification is required.  Points in low-density embedding regions are
labelled as noise and excluded from communities.  The spectral embedding
translates the HVG's chain-like topology into a low-dimensional space where
natural cluster structure can be detected without imposing a partition count.
- **HDBSCAN-Geographic**: same HDBSCAN algorithm applied to projected $(x, y)$
node coordinates in kilometres (EPSG:32632); no graph structure is used.
Because the HVG encodes temporal rather than spatial structure, divergence
between HDBSCAN-Spectral and HDBSCAN-Geographic is expected to be large —
confirming that HVG communities encode temporal seismic episodes, not merely
geographic clusters.

**Overlapping-community method**
- **BigCLAM** (Yang & Leskovec 2013): solves for an $N \times k$ affiliation
matrix $F$ via non-negative matrix factorisation; the hard partition is recovered
by $\arg\max_c F_{ic}$.  Events immediately adjacent to major mainshocks
(amplitude barriers) may carry soft affiliation in two seismic episodes,
capturing the transition between episodes.

**Partition quality scoring**
All seven partitions are evaluated on 9 quality metrics — modularity, conductance,
coverage, Ncut, map equation, DC-SBM log-likelihood, Surprise, geographic
compactness, and depth coherence — via `compare_partitions`.

*References:* Newman & Girvan (2004) *Phys. Rev. E* 69, 026113;
Blondel et al. (2008) *J. Stat. Mech.* P10008;
Rosvall & Bergstrom (2008) *PNAS* 105, 1118;
Campello et al. (2013) *ECML-PKDD* 160–175;
Yang & Leskovec (2013) *ACM TKDD* 7(3), 1–42.

**Expected output:** 5–20 communities (fewer than NVG because fewer inter-episode
links exist to confuse the algorithm); NMI matrix entries $\geq 0.7$ between
Louvain and Consensus Louvain; Spectral may differ more due to the path-like
topology; HDBSCAN-Geographic should differ markedly from HDBSCAN-Spectral,
confirming that HVG communities encode temporal episodes rather than spatial zones.

In [ ]:
print("Running community detection on HVG graph…")

print("Louvain…")
community_map = run_louvain(G, seed=42)
k_louvain     = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G, community_map,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

print("Consensus Louvain (100 runs)…")
community_map_consensus = run_consensus_louvain(G, n_runs=100, seed=42)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G, community_map_consensus,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

print(f"Spectral (k={min(k_louvain, 8)})…")
_k_spec = min(k_louvain, 8)
community_map_spectral = run_spectral(G, k=_k_spec, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities")
plot_community_geo(
    G, community_map_spectral,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Spectral",
)

print("InfoMap (undirected)…")
community_map_infomap = run_infomap(G, directed=False, seed=42)
print(f"  {len(set(community_map_infomap.values()))} communities")
plot_community_geo(
    G, community_map_infomap,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="InfoMap",
)

partitions = {
    "Louvain":   community_map,
    "Consensus": community_map_consensus,
    "Spectral":  community_map_spectral,
    "InfoMap":   community_map_infomap,
}
print("Computing NMI…")
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="HVG Italy")

print("HDBSCAN-Spectral…")
community_map_hdbscan_spec = run_hdbscan_spectral(G, min_cluster_size=10, seed=42)
print(f"  {len(set(community_map_hdbscan_spec.values()))} clusters")
plot_community_geo(
    G, community_map_hdbscan_spec,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Spectral",
)

print("HDBSCAN-Geographic…")
community_map_hdbscan_geo = run_hdbscan_geo(G, min_cluster_size=10)
print(f"  {len(set(community_map_hdbscan_geo.values()))} clusters")
plot_community_geo(
    G, community_map_hdbscan_geo,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Geographic",
)

print("BigCLAM…")
F_bigclam, community_map_bigclam = run_bigclam(
    G, k=k_louvain, n_iter=100, lr=0.005, seed=42,
)
print(f"  {len(set(community_map_bigclam.values()))} non-empty communities")
plot_community_geo(
    G, community_map_bigclam,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="BigCLAM",
)

partitions_ext = {
    **partitions,
    "HDBSCAN-Spec": community_map_hdbscan_spec,
    "HDBSCAN-Geo":  community_map_hdbscan_geo,
    "BigCLAM":      community_map_bigclam,
}
nmi_ext = compute_nmi_matrix(partitions_ext)
display(nmi_ext.round(3))
plot_nmi_heatmap(nmi_ext, title="HVG Italy — all methods")

scores_df = compare_partitions(G, partitions_ext, cell_size_km=10.0)
display(scores_df.round(4))
plot_partition_scores(scores_df, title="HVG Italy")

## Directed Community Detection

The HVG is undirected, but time provides a natural causal orientation: edge
$i - j$ (with $i < j$) is directed $i \to j$ (earlier event influences later).
The directed HVG encodes **horizontal amplitude causality**: which past events
maintain unobstructed horizontal lines of sight to future events.

Directed Louvain via the Leiden algorithm (Traag, Waltman & van Eck 2019)
maximises the **directed modularity** (Leicht & Newman 2008):

$$Q_d = \frac{1}{m}\sum_{ij}\left[A_{ij} - \frac{k_i^{\text{out}} k_j^{\text{in}}}{m}\right]\delta(c_i, c_j)$$

**NMI between directed and undirected Louvain** measures whether the time arrow
reorganises community assignments.  High NMI implies the amplitude structure is
approximately symmetric (no preferred direction of causality) — consistent with
a stationary magnitude series.  Low NMI implies foreshock ramps vs aftershock
decays partition the sequence differently when flow direction is respected.

**Expected output:** Directed and undirected Louvain partitions likely agree
closely (NMI $> 0.7$) because the HVG topology is close to a bidirectional
path graph — the time arrow primarily converts undirected edges to directed
ones without fundamentally altering the community structure.

In [ ]:
print("Building directed HVG (time-arrow orientation: i→j for i<j)…")
G_dir = nx.DiGraph()
G_dir.add_nodes_from(G.nodes(data=True))
G_dir.add_edges_from(
    (u, v) if u < v else (v, u)
    for u, v in G.edges()
)
print(f"Directed HVG: {G_dir.number_of_nodes():,} nodes, {G_dir.number_of_edges():,} edges")

print("Running directed Louvain (Leiden) on directed HVG…")
community_map_directed = run_directed_louvain(G_dir, seed=42)
k_directed = len(set(community_map_directed.values()))
print(f"  {k_directed} directed communities  (vs {k_louvain} undirected)")

plot_community_geo(
    G_dir, community_map_directed,
    title="HVG Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Directed Louvain (Leiden)",
)

partitions_full = {**partitions, "Directed Louvain": community_map_directed}
nmi_full = compute_nmi_matrix(partitions_full)
display(nmi_full.round(3))
plot_nmi_heatmap(nmi_full, title="HVG Italy — directed vs undirected")

## Community Markov Flow

The Louvain partition coarse-grains the directed HVG into a $K \times K$
flow matrix $F_{ij} = \sum_{u \in C_i,\, v \in C_j} w_{uv}$, counting directed
visibility links crossing from community $i$ to community $j$.

Row-normalisation gives the **Markov transition matrix**:

$$T_{ij} = \frac{F_{ij}}{\sum_j F_{ij}}$$

$T_{ij}$ has a temporal reading: it is the fraction of time that the magnitude
sequence crosses a community boundary from seismic episode $i$ to episode $j$.
**High self-retention** ($T_{ii} \approx 1$) indicates that communities are
amplitude-isolated — separated by magnitude barriers that block most long-range
horizontal visibility.  **High off-diagonal entries** identify pairs of episodes
frequently adjacent in the time series, corresponding to fore- and aftershock
sequences of the same mainshock.

The **stationary distribution** $\boldsymbol{\pi}$ (dominant left eigenvector of $T$)
gives the long-run fraction of time the visibility walk spends in each community —
a measure of the relative duration and connectivity of each seismic episode.

*Reference:* Rosvall, M., & Bergstrom, C. T. (2008).
Maps of random walks on complex networks reveal community structure. *PNAS*, 105(4), 1118–1123.

**Expected output:** $K \approx 5$–$20$ community states.  Self-retention entries
$T_{ii} \approx 0.85$–$0.98$ (high, because the HVG is sparse and inter-community
links are few).  The stationary distribution likely concentrates in the 2–3 communities
covering the longest quiescent periods between major mainshocks.

In [ ]:
from src.community_flow import (
    build_community_flow_matrix,
    compute_markov_chain,
    compute_stationary_distribution,
    community_flow_stats,
    plot_flow_heatmap,
    plot_flow_entropy,
    plot_stationary_distribution,
)

print("Building community flow matrix (Louvain partition, directed HVG)…")
flow_count_df = build_community_flow_matrix(G_dir, community_map)
T_markov  = compute_markov_chain(flow_count_df)
stat_dist = compute_stationary_distribution(T_markov)
df_flow   = community_flow_stats(T_markov, flow_count_df, stat_dist)

K_comm = T_markov.shape[0]
print(f"Markov chain: {K_comm} community states")
print(f"Mean self-retention:  {df_flow['self_retention'].mean():.3f}")
print(f"Mean outflow entropy: {df_flow['entropy'].mean():.3f} bits "
      f"(max = log₂({K_comm}) = {np.log2(K_comm):.3f})")
print(f"Dominant attractor:   Community {df_flow.iloc[0]['community']} "
      f"(π = {df_flow.iloc[0]['stationary']:.4f})")
display(df_flow)

plot_flow_heatmap(T_markov, flow_count_df, title="HVG Italy")
plot_flow_entropy(df_flow, K=K_comm, title="HVG Italy")
plot_stationary_distribution(df_flow, title="HVG Italy")

out_path = RESULTS_DIR / "data" / "italy_hvg_community_flow.csv"
df_flow.to_csv(out_path, index=False)
print(f"Community flow stats saved → {out_path}")

## Assortativity

Newman (2003) assortativity $r$ quantifies whether horizontal visibility links
preferentially connect events with similar attributes (magnitude, depth, degree).

$$r = \frac{\sum_{jk} jk (e_{jk} - q_j q_k)}{\sigma_q^2}$$

where $e_{jk}$ is the joint degree distribution and $q_k$ is the remaining-degree
distribution ($r = 1$ perfectly assortative, $r = -1$ perfectly disassortative).

**Magnitude assortativity** $r > 0$ is more likely in the HVG than in the NVG:
the stricter horizontal condition filters out low-magnitude neighbouring events,
leaving primarily mainshock-to-mainshock long-range links where both endpoints
have similar (large) magnitudes.

**Degree assortativity** $r < 0$ is expected if the few high-degree mainshock
hubs connect primarily to low-degree (degree-2) aftershocks.  This reflects the
asymmetric amplitude structure of aftershock sequences: the mainshock dominates
its neighbourhood, collecting links from many small events that are otherwise
invisible beyond one step.

**Depth assortativity** may be weak or near zero, as depth does not affect HVG
edge formation (only magnitude matters).

*Reference:* Newman, M. E. J. (2003). Mixing patterns in networks. *Physical Review E*, 67(2), 026126.

**Expected output:** Magnitude $r \approx -0.05$ to $+0.15$ (near-zero to mildly
positive); degree $r \approx -0.3$ to $-0.6$ (strongly disassortative, hub-spoke
structure more pronounced than in NVG).

In [ ]:
print("Computing assortativity (node attributes pre-attached at construction)…")
df_assort = compute_assortativity(G)
display(df_assort)
plot_assortativity(G, title="HVG Italy")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G, title="HVG Italy", gamma=gamma_hvg)

print("Rich-club coefficient:")
plot_rich_club(G, title="HVG Italy")

## Export Results

Centralisation metrics, community assignments, and the network object are
persisted for downstream comparison with the ABE, BP, TL, and ZBZ networks.

**CSV** (`italy_hvg_network_metrics.csv`): one row per node with all centrality
scores and Louvain community label.  Use this to compare HVG hub rankings against
the other four models.

**Pickle** (`italy_hvg.pkl`): serialised NetworkX `Graph` object with all node
attributes (lat, lon, depth, magnitude, degree).  Load with
`pickle.load(open("results/cache/italy_hvg.pkl", "rb"))` for fast graph access
in the comparison notebook without re-running the full build.

**GEXF** (`italy_hvg.gexf`): Gephi-compatible export for visual exploration.
Suggested layout: ForceAtlas2 (fast, handles sparse graphs well).
Colour by `community` (categorical) or `magnitude` (sequential plasma).
The path-like backbone of the HVG will be visible as a temporal spine.

**Expected output:** CSV with ~215,000 rows × 10 columns; pickle ~50 MB; GEXF ~200 MB.

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "italy_hvg_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved → {out_path}  ({len(df_final):,} rows)")

pkl_path = RESULTS_DIR / "cache" / "italy_hvg.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(G, f)
print(f"Network cached → {pkl_path}")

gexf_path = RESULTS_DIR / "gephi" / "italy_hvg.gexf"
nx.write_gexf(G, gexf_path)
print(f"Gephi export → {gexf_path}")